# Pipeline testing — capture as many overtakes as possible

This notebook replaces the earlier failed draft and tries to recover **as many genuine overtakes as possible** from FastF1 race data.

## Main fixes vs the previous attempt

- use **`EventName`** instead of `Country` to load sessions, so races like **Miami / Las Vegas / Emilia Romagna** are not lost,
- include **all actual race weekends** (`RoundNumber > 0`) instead of dropping sprint weekends,
- keep a **per-race audit table** so missing races are visible,
- extract **raw pairwise order flips between consecutive laps** as the widest lap-level approximation,
- then tune filters to get closer to published season totals.

## Important limitation

This is still a **lap-end** approximation. It can miss:

- overtakes and repasses inside the same lap,
- passes only visible in higher-frequency timing,
- some classification artifacts around pits / retirements / safety cars.

So this notebook is meant as a **better counting experiment**, not a perfect official overtake counter.

In [1]:
import warnings
from itertools import combinations, product
from pathlib import Path
import sys

import fastf1
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipeline import fastf1_utils as ffu

CACHE_DIR = ROOT / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
fastf1.Cache.enable_cache(str(CACHE_DIR))

REFERENCE_TOTALS = {
    2022: 785,
    2023: 858,
    2024: 788,
    # 2025: add a published season total here when you want error columns for that year
}

DEFAULT_YEARS = [2022, 2023, 2024, 2025]

print("Cache:", CACHE_DIR)
print("Reference totals:", REFERENCE_TOTALS)

Cache: /Users/aminnami/Desktop/hamid/ai_in_industry/formula1-overtake-prediction/cache
Reference totals: {2022: 785, 2023: 858, 2024: 788}


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Extraction strategy

For each race:

1. Load the **Race** session using the schedule row's **`EventName`**.
2. Build a lap-level table of driver positions.
3. For each pair of drivers present on consecutive laps, mark a **raw flip event** when their relative order reverses from lap `L` to lap `L+1`.
4. Store rich metadata for later filtering:
   - pit involvement,
   - safety car / yellow flags,
   - position jump size,
   - whether the pair was adjacent before or after the pass,
   - timing accuracy.
5. Compare filtered totals with published season references.

This raw-flip approach intentionally aims wide: it will catch more possible overtakes than the existing battle detector, then let us trim obvious artifacts afterward.

In [2]:
def race_event_rows(year: int) -> pd.DataFrame:
    schedule = fastf1.get_event_schedule(year)
    out = schedule.copy()
    out = out[out["RoundNumber"].fillna(0).astype(int) > 0].copy()
    return out.reset_index(drop=True)


def load_race_session(year: int, event_name: str):
    return ffu.load_session(year=year, gp=event_name, identifier="R", cache_path=str(CACHE_DIR))


def session_year(session, fallback_year: int) -> int:
    try:
        val = session.event.get("Year")
        if val is not None and not pd.isna(val):
            return int(val)
    except Exception:
        pass
    try:
        dt = session.event.get("EventDate")
        if dt is not None and not pd.isna(dt):
            return int(pd.Timestamp(dt).year)
    except Exception:
        pass
    return int(fallback_year)


def lap_frame(session) -> pd.DataFrame:
    cols = [
        "Driver",
        "LapNumber",
        "Position",
        "LapTime",
        "PitInTime",
        "PitOutTime",
        "LapStartTime",
        "Time",
        "TrackStatus",
        "IsAccurate",
    ]
    present = [c for c in cols if c in session.laps.columns]
    df = session.laps[present].copy()
    df = df[df["Driver"].notna() & df["LapNumber"].notna() & df["Position"].notna()].copy()
    df["Driver"] = df["Driver"].astype(str)
    df["LapNumber"] = df["LapNumber"].astype(int)
    df["Position"] = df["Position"].astype(int)
    if "IsAccurate" in df.columns:
        df["IsAccurate"] = df["IsAccurate"].fillna(False).astype(bool)
    else:
        df["IsAccurate"] = True
    return df.sort_values(["LapNumber", "Position", "Driver"]).reset_index(drop=True)


def pit_related(row: pd.Series) -> bool:
    return pd.notna(row.get("PitInTime")) or pd.notna(row.get("PitOutTime"))


def extract_raw_overtake_candidates(session, fallback_year: int) -> pd.DataFrame:
    laps = lap_frame(session)
    if laps.empty:
        return pd.DataFrame()

    year = session_year(session, fallback_year)
    race_name = str(session.event.get("EventName") or "Unknown")
    round_number = int(session.event.get("RoundNumber") or 0)
    event_format = str(session.event.get("EventFormat") or "")
    max_lap = int(laps["LapNumber"].max())

    rows: list[dict] = []
    for lap in range(1, max_lap):
        cur = laps[laps["LapNumber"] == lap].set_index("Driver", drop=False)
        nxt = laps[laps["LapNumber"] == lap + 1].set_index("Driver", drop=False)
        shared = sorted(set(cur.index) & set(nxt.index))
        if len(shared) < 2:
            continue

        safety_car, yellow_flag = ffu.detect_safety_car_and_flags(session, lap + 1)

        for i, a in enumerate(shared):
            for b in shared[i + 1 :]:
                a0, b0 = cur.loc[a], cur.loc[b]
                a1, b1 = nxt.loc[a], nxt.loc[b]
                pa0, pb0 = int(a0["Position"]), int(b0["Position"])
                pa1, pb1 = int(a1["Position"]), int(b1["Position"])

                if pa0 == pb0 or pa1 == pb1:
                    continue
                if (pa0 - pb0) * (pa1 - pb1) >= 0:
                    continue

                if pa0 > pb0 and pa1 < pb1:
                    overtaker, overtaken = a, b
                    ov0, ov1, od0, od1 = a0, a1, b0, b1
                    overtaker_prev, overtaker_next = pa0, pa1
                    overtaken_prev, overtaken_next = pb0, pb1
                elif pb0 > pa0 and pb1 < pa1:
                    overtaker, overtaken = b, a
                    ov0, ov1, od0, od1 = b0, b1, a0, a1
                    overtaker_prev, overtaker_next = pb0, pb1
                    overtaken_prev, overtaken_next = pa0, pa1
                else:
                    continue

                rows.append(
                    {
                        "year": year,
                        "race_name": race_name,
                        "round_number": round_number,
                        "event_format": event_format,
                        "lap_number": lap + 1,
                        "overtaker": overtaker,
                        "overtaken": overtaken,
                        "overtaker_prev_pos": overtaker_prev,
                        "overtaker_next_pos": overtaker_next,
                        "overtaken_prev_pos": overtaken_prev,
                        "overtaken_next_pos": overtaken_next,
                        "position_gain": int(overtaker_prev - overtaker_next),
                        "consecutive_before": bool(overtaker_prev == overtaken_prev + 1),
                        "consecutive_after": bool(overtaken_next == overtaker_next + 1),
                        "pit_related": bool(
                            pit_related(ov0) or pit_related(ov1) or pit_related(od0) or pit_related(od1)
                        ),
                        "accurate_timing": bool(
                            ov0.get("IsAccurate", True)
                            and ov1.get("IsAccurate", True)
                            and od0.get("IsAccurate", True)
                            and od1.get("IsAccurate", True)
                        ),
                        "safety_car": bool(safety_car),
                        "yellow_flag": bool(yellow_flag),
                        "lap1_or_restart_like": bool(lap + 1 <= 2),
                    }
                )

    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.sort_values(["year", "round_number", "lap_number", "overtaker", "overtaken"]).reset_index(drop=True)

In [3]:
_SESSION_CACHE: dict[tuple[int, str], object] = {}
_RAW_CACHE: dict[tuple[int, str], pd.DataFrame] = {}


def get_or_load_session(year: int, event_name: str):
    key = (year, event_name)
    if key not in _SESSION_CACHE:
        _SESSION_CACHE[key] = load_race_session(year, event_name)
    return _SESSION_CACHE[key]


def collect_season_raw_events(year: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    frames: list[pd.DataFrame] = []
    audit_rows: list[dict] = []

    events = race_event_rows(year)
    for _, event in tqdm(events.iterrows(), total=len(events), desc=f"Season {year}"):
        event_name = str(event["EventName"])
        event_format = str(event.get("EventFormat") or "")
        try:
            key = (year, event_name)
            if key in _RAW_CACHE:
                raw = _RAW_CACHE[key].copy()
            else:
                session = get_or_load_session(year, event_name)
                raw = extract_raw_overtake_candidates(session, year)
                _RAW_CACHE[key] = raw.copy()

            audit_rows.append(
                {
                    "year": year,
                    "round_number": int(event.get("RoundNumber") or 0),
                    "event_name": event_name,
                    "event_format": event_format,
                    "status": "ok",
                    "raw_events": int(len(raw)),
                    "error": None,
                }
            )
            if not raw.empty:
                frames.append(raw)
        except Exception as exc:
            audit_rows.append(
                {
                    "year": year,
                    "round_number": int(event.get("RoundNumber") or 0),
                    "event_name": event_name,
                    "event_format": event_format,
                    "status": "error",
                    "raw_events": 0,
                    "error": str(exc),
                }
            )

    raw_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    audit_df = pd.DataFrame(audit_rows).sort_values(["round_number", "event_name"]).reset_index(drop=True)
    return raw_df, audit_df


def season_audit_summary(audit_df: pd.DataFrame) -> pd.DataFrame:
    if audit_df.empty:
        return audit_df.copy()
    return pd.DataFrame(
        {
            "expected_races": [int(len(audit_df))],
            "loaded_ok": [int((audit_df["status"] == "ok").sum())],
            "failed": [int((audit_df["status"] == "error").sum())],
            "races_with_zero_events": [int(((audit_df["status"] == "ok") & (audit_df["raw_events"] == 0)).sum())],
            "total_raw_events": [int(audit_df["raw_events"].sum())],
        }
    )

In [4]:
def apply_filters(
    df: pd.DataFrame,
    *,
    exclude_pit_related: bool = True,
    exclude_lap1: bool = True,
    exclude_safety_car: bool = True,
    exclude_yellow_flag: bool = False,
    require_accurate_timing: bool = False,
    adjacency_rule: str = "none",  # none / before / after / either
    max_position_gain: int | None = None,
) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    out = df.copy()
    if exclude_pit_related:
        out = out[~out["pit_related"]]
    if exclude_lap1:
        out = out[~out["lap1_or_restart_like"]]
    if exclude_safety_car:
        out = out[~out["safety_car"]]
    if exclude_yellow_flag:
        out = out[~out["yellow_flag"]]
    if require_accurate_timing:
        out = out[out["accurate_timing"]]
    if adjacency_rule == "before":
        out = out[out["consecutive_before"]]
    elif adjacency_rule == "after":
        out = out[out["consecutive_after"]]
    elif adjacency_rule == "either":
        out = out[out["consecutive_before"] | out["consecutive_after"]]
    if max_position_gain is not None:
        out = out[out["position_gain"] <= max_position_gain]
    return out.reset_index(drop=True)


def compare_to_reference(season_raw: dict[int, pd.DataFrame], **filter_kwargs) -> pd.DataFrame:
    rows = []
    for year, raw in season_raw.items():
        filtered = apply_filters(raw, **filter_kwargs)
        ref = REFERENCE_TOTALS.get(year)
        observed = int(len(filtered))
        rows.append(
            {
                "year": year,
                "raw_candidates": int(len(raw)),
                "filtered_events": observed,
                "reference_total": ref,
                "signed_error": None if ref is None else observed - int(ref),
                "abs_error": None if ref is None else abs(observed - int(ref)),
            }
        )
    return pd.DataFrame(rows).sort_values("year").reset_index(drop=True)


def filter_grid_search(season_raw: dict[int, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for exclude_pit_related, exclude_lap1, exclude_safety_car, adjacency_rule, max_position_gain in product(
        [True, False],
        [True, False],
        [True, False],
        ["none", "before", "after", "either"],
        [1, 2, 3, None],
    ):
        comp = compare_to_reference(
            season_raw,
            exclude_pit_related=exclude_pit_related,
            exclude_lap1=exclude_lap1,
            exclude_safety_car=exclude_safety_car,
            exclude_yellow_flag=False,
            require_accurate_timing=False,
            adjacency_rule=adjacency_rule,
            max_position_gain=max_position_gain,
        )
        known = comp[comp["reference_total"].notna()].copy()
        rows.append(
            {
                "exclude_pit_related": exclude_pit_related,
                "exclude_lap1": exclude_lap1,
                "exclude_safety_car": exclude_safety_car,
                "adjacency_rule": adjacency_rule,
                "max_position_gain": max_position_gain,
                "total_abs_error": int(known["abs_error"].sum()),
            }
        )
    return pd.DataFrame(rows).sort_values("total_abs_error").reset_index(drop=True)

In [5]:
YEARS = DEFAULT_YEARS

season_raw: dict[int, pd.DataFrame] = {}
season_audit: dict[int, pd.DataFrame] = {}

for year in YEARS:
    raw_df, audit_df = collect_season_raw_events(year)
    season_raw[year] = raw_df
    season_audit[year] = audit_df

    print(f"\n=== {year} ===")
    display(season_audit_summary(audit_df))
    display(audit_df[["round_number", "event_name", "event_format", "status", "raw_events", "error"]])
    print(f"Raw candidate events: {len(raw_df):,}")
    if not raw_df.empty:
        print(f"Races with raw events: {raw_df['race_name'].nunique()}")
        display(raw_df.head(20))

Season 2022:   0%|          | 0/22 [00:00<?, ?it/s]

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.7.0]


req            INFO 	Using cached data for session_info


req            INFO 	Using cached data for driver_info


req            INFO 	Using cached data for session_status_data


...


=== 2025 ===


,expected_races,loaded_ok,failed,races_with_zero_events,total_raw_events
0,24,24,0,0,3906


,round_number,event_name,event_format,status,raw_events,error
0,1,Australian Grand Prix,conventional,ok,94,None
1,2,Chinese Grand Prix,sprint_qualifying,ok,174,None
2,3,Japanese Grand Prix,conventional,ok,125,None
3,4,Bahrain Grand Prix,conventional,ok,302,None
4,5,Saudi Arabian Grand Prix,conventional,ok,110,None
5,6,Miami Grand Prix,sprint_qualifying,ok,78,None
6,7,Emilia Romagna Grand Prix,conventional,ok,188,None
7,8,Monaco Grand Prix,conventional,ok,81,None
8,9,Spanish Grand Prix,conventional,ok,211,None
9,10,Canadian Grand Prix,conventional,ok,199,None


Raw candidate events: 3,906
Races with raw events: 24


,year,race_name,round_number,event_format,lap_number,overtaker,overtaken,overtaker_prev_pos,overtaker_next_pos,overtaken_prev_pos,overtaken_next_pos,position_gain,consecutive_before,consecutive_after,pit_related,accurate_timing,safety_car,yellow_flag,lap1_or_restart_like
0,2025,Australian Grand Prix,1,conventional,14,ANT,HUL,13,12,12,13,1,True,True,False,True,False,False,False
1,2025,Australian Grand Prix,1,conventional,16,HUL,ANT,13,12,12,13,1,True,True,False,True,False,False,False
2,2025,Australian Grand Prix,1,conventional,17,ANT,HUL,13,12,12,13,1,True,True,False,True,False,False,False
3,2025,Australian Grand Prix,1,conventional,17,PIA,VER,3,2,2,3,1,True,True,False,True,False,False,False
4,2025,Australian Grand Prix,1,conventional,19,LAW,OCO,16,15,15,16,1,True,True,False,True,False,False,False
5,2025,Australian Grand Prix,1,conventional,22,ANT,STR,12,11,11,12,1,True,True,False,True,False,False,False
6,2025,Australian Grand Prix,1,conventional,33,OCO,LAW,16,14,15,15,2,True,True,True,False,False,False,False
7,2025,Australian Grand Prix,1,conventional,34,BEA,LAW,16,15,15,16,1,True,True,True,False,False,False,False
8,2025,Australian Grand Prix,1,conventional,39,LAW,BEA,16,14,15,16,2,True,False,True,False,False,False,False
9,2025,Australian Grand Prix,1,conventional,39,LAW,OCO,16,14,14,15,2,False,True,True,False,False,False,False


In [6]:
grid = filter_grid_search(season_raw)
display(grid.head(20))

best = grid.iloc[0].to_dict()
best_filters = {
    "exclude_pit_related": bool(best["exclude_pit_related"]),
    "exclude_lap1": bool(best["exclude_lap1"]),
    "exclude_safety_car": bool(best["exclude_safety_car"]),
    "exclude_yellow_flag": False,
    "require_accurate_timing": False,
    "adjacency_rule": str(best["adjacency_rule"]),
    "max_position_gain": None if pd.isna(best["max_position_gain"]) else int(best["max_position_gain"]),
}

comparison = compare_to_reference(season_raw, **best_filters)
print("Chosen filters:", best_filters)
display(comparison)

for year in YEARS:
    sample = apply_filters(season_raw[year], **best_filters)
    print(f"\nSample filtered events for {year}: {len(sample):,}")
    display(sample.head(20))

,exclude_pit_related,exclude_lap1,exclude_safety_car,adjacency_rule,max_position_gain,total_abs_error
0,True,False,False,none,NaN,53
1,True,False,True,none,NaN,53
2,True,False,False,none,3.0,58
3,True,False,True,none,3.0,58
4,True,True,True,none,NaN,119
5,True,True,False,none,NaN,119
6,True,False,False,none,2.0,120
7,True,False,True,none,2.0,120
8,True,True,True,none,3.0,130
9,True,True,False,none,3.0,130


Chosen filters: {'exclude_pit_related': True, 'exclude_lap1': False, 'exclude_safety_car': False, 'exclude_yellow_flag': False, 'require_accurate_timing': False, 'adjacency_rule': 'none', 'max_position_gain': None}


,year,raw_candidates,filtered_events,reference_total,signed_error,abs_error
0,2022,3580,794,785.0,9.0,9.0
1,2023,4158,834,858.0,-24.0,24.0
2,2024,3785,768,788.0,-20.0,20.0
3,2025,3906,746,NaN,NaN,NaN



Sample filtered events for 2022: 794


,year,race_name,round_number,event_format,lap_number,overtaker,overtaken,overtaker_prev_pos,overtaker_next_pos,overtaken_prev_pos,overtaken_next_pos,position_gain,consecutive_before,consecutive_after,pit_related,accurate_timing,safety_car,yellow_flag,lap1_or_restart_like
0,2022,Bahrain Grand Prix,1,conventional,2,STR,NOR,17,16,16,17,1,True,True,False,False,False,False,True
1,2022,Bahrain Grand Prix,1,conventional,2,ZHO,LAT,19,18,18,19,1,True,True,False,False,False,False,True
2,2022,Bahrain Grand Prix,1,conventional,3,PER,MAG,6,5,5,6,1,True,True,False,True,False,False,False
3,2022,Bahrain Grand Prix,1,conventional,3,ZHO,NOR,18,17,17,18,1,True,True,False,True,False,False,False
4,2022,Bahrain Grand Prix,1,conventional,4,ZHO,STR,17,16,16,17,1,True,True,False,True,False,False,False
5,2022,Bahrain Grand Prix,1,conventional,5,RUS,MAG,7,6,6,7,1,True,True,False,True,False,False,False
6,2022,Bahrain Grand Prix,1,conventional,5,TSU,ALB,12,11,11,12,1,True,True,False,True,False,False,False
7,2022,Bahrain Grand Prix,1,conventional,5,ZHO,HUL,16,15,15,16,1,True,True,False,True,False,False,False
8,2022,Bahrain Grand Prix,1,conventional,6,BOT,MSC,14,13,13,14,1,True,True,False,True,False,False,False
9,2022,Bahrain Grand Prix,1,conventional,6,RIC,LAT,20,19,19,20,1,True,True,False,True,False,False,False


...

,year,race_name,round_number,event_format,lap_number,overtaker,overtaken,overtaker_prev_pos,overtaker_next_pos,overtaken_prev_pos,overtaken_next_pos,position_gain,consecutive_before,consecutive_after,pit_related,accurate_timing,safety_car,yellow_flag,lap1_or_restart_like
0,2023,Bahrain Grand Prix,1,conventional,2,GAS,MAG,19,18,18,19,1,True,True,False,False,False,False,True
1,2023,Bahrain Grand Prix,1,conventional,3,DEV,MAG,20,19,19,20,1,True,True,False,True,False,False,False
2,2023,Bahrain Grand Prix,1,conventional,4,ALB,NOR,12,11,10,12,1,False,True,False,True,False,False,False
3,2023,Bahrain Grand Prix,1,conventional,4,OCO,NOR,11,10,10,12,1,True,False,False,True,False,False,False
4,2023,Bahrain Grand Prix,1,conventional,4,TSU,HUL,15,14,14,15,1,True,True,False,True,False,False,False
5,2023,Bahrain Grand Prix,1,conventional,5,STR,BOT,9,8,8,9,1,True,True,False,True,False,False,False
6,2023,Bahrain Grand Prix,1,conventional,14,BOT,MAG,10,8,9,9,2,True,True,False,True,False,False,False
7,2023,Bahrain Grand Prix,1,conventional,17,ALO,BOT,7,6,6,7,1,True,True,False,True,False,False,False
8,2023,Bahrain Grand Prix,1,conventional,18,RUS,BOT,8,7,7,8,1,True,True,False,True,False,False,False
9,2023,Bahrain Grand Prix,1,conventional,18,ZHO,HUL,17,15,16,16,2,True,True,False,True,False,False,False



Sample filtered events for 2024: 768


,year,race_name,round_number,event_format,lap_number,overtaker,overtaken,overtaker_prev_pos,overtaker_next_pos,overtaken_prev_pos,overtaken_next_pos,position_gain,consecutive_before,consecutive_after,pit_related,accurate_timing,safety_car,yellow_flag,lap1_or_restart_like
0,2024,Bahrain Grand Prix,1,conventional,2,SAR,RIC,15,14,14,15,1,True,True,False,False,False,False,True
1,2024,Bahrain Grand Prix,1,conventional,3,NOR,ALO,7,6,6,7,1,True,True,False,True,False,False,False
2,2024,Bahrain Grand Prix,1,conventional,3,RUS,LEC,3,2,2,3,1,True,True,False,True,False,False,False
3,2024,Bahrain Grand Prix,1,conventional,5,PIA,ALO,8,7,7,8,1,True,True,False,True,False,False,False
4,2024,Bahrain Grand Prix,1,conventional,5,STR,BOT,19,18,18,19,1,True,True,False,True,False,False,False
5,2024,Bahrain Grand Prix,1,conventional,7,PER,LEC,4,3,3,4,1,True,True,False,True,False,False,False
6,2024,Bahrain Grand Prix,1,conventional,10,HAM,ALO,9,8,8,9,1,True,True,False,True,False,False,False
7,2024,Bahrain Grand Prix,1,conventional,17,SAI,LEC,5,4,4,5,1,True,True,False,True,False,False,False
8,2024,Bahrain Grand Prix,1,conventional,17,TSU,STR,12,11,11,12,1,True,True,False,True,False,False,False
9,2024,Bahrain Grand Prix,1,conventional,18,SAI,RUS,4,3,3,4,1,True,True,False,True,False,False,False



Sample filtered events for 2025: 746


,year,race_name,round_number,event_format,lap_number,overtaker,overtaken,overtaker_prev_pos,overtaker_next_pos,overtaken_prev_pos,overtaken_next_pos,position_gain,consecutive_before,consecutive_after,pit_related,accurate_timing,safety_car,yellow_flag,lap1_or_restart_like
0,2025,Australian Grand Prix,1,conventional,14,ANT,HUL,13,12,12,13,1,True,True,False,True,False,False,False
1,2025,Australian Grand Prix,1,conventional,16,HUL,ANT,13,12,12,13,1,True,True,False,True,False,False,False
2,2025,Australian Grand Prix,1,conventional,17,ANT,HUL,13,12,12,13,1,True,True,False,True,False,False,False
3,2025,Australian Grand Prix,1,conventional,17,PIA,VER,3,2,2,3,1,True,True,False,True,False,False,False
4,2025,Australian Grand Prix,1,conventional,19,LAW,OCO,16,15,15,16,1,True,True,False,True,False,False,False
5,2025,Australian Grand Prix,1,conventional,22,ANT,STR,12,11,11,12,1,True,True,False,True,False,False,False
6,2025,Australian Grand Prix,1,conventional,43,TSU,LEC,6,5,5,6,1,True,True,False,True,False,False,False
7,2025,Australian Grand Prix,1,conventional,44,GAS,LEC,9,4,6,5,5,False,True,False,True,False,False,False
8,2025,Australian Grand Prix,1,conventional,44,HAM,LEC,8,3,6,5,5,False,False,False,True,False,False,False
9,2025,Australian Grand Prix,1,conventional,45,HAM,TSU,3,2,2,3,1,True,True,False,True,False,False,False


## How to read the output

- **Raw candidate events**: widest lap-level estimate of possible overtakes.
- **Audit table**: confirms whether every race loaded and whether any races still return zero events.
- **Grid search**: tries filter combinations to match reference totals as closely as possible.
- **Filtered events**: the current best approximation, not an official count.

If totals are still far off, the next step is to use **higher-frequency timing** than lap-end positions, because same-lap pass/repass sequences are invisible here.